# Exemplar-guided population refinement

A headless, executable example using synthetic cells. It demonstrates explicit
classes, optional per-marker agreement, case/ROI sampling, the existing NapariSBT
image feature worker, grouped validation and non-destructive AnnData integration.

The synthetic data demonstrate API behaviour only. Replace the setup with your
AnnData, masks, images and transferred labels for scientific use. See the
population-refinement guide for assumptions and review requirements.

In [1]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import anndata as ad
import tifffile
from IPython.display import display
from SpatialBiologyToolkit import population_refinement as pr

# A disposable workspace keeps generated example assets out of the repository.
workspace = Path(tempfile.mkdtemp(prefix="sbt_refinement_example_"))
images, masks = workspace / "images", workspace / "masks"
images.mkdir(); masks.mkdir()
rng = np.random.default_rng(42)
rows = []
for case in range(3):
    for roi_number in range(2):
        roi = f"case{case}_roi{roi_number}"
        mask = np.zeros((64, 64), dtype=np.uint16)
        channels = {name: rng.uniform(0, .03, mask.shape).astype("float32") for name in ("IBA1", "SOX2")}
        (images / roi).mkdir()
        for i in range(16):
            y, x = 3 + (i // 4)*15, 3 + (i % 4)*15
            mask[y:y+8, x:x+8] = i+1
            label = "Myeloid" if i % 2 else "Glial"
            for marker in channels:
                positive = (marker == "IBA1") == (label == "Myeloid")
                channels[marker][y:y+8, x:x+8] += (.8 if positive else .1) + rng.normal(0, .015, (8,8))
            rows.append(dict(ROI=roi, Case=f"case{case}", ObjectNumber=i+1, parent="mixed",
                             atlas_label=label, maxfuse_score=float(rng.uniform(-.1, .9))))
        tifffile.imwrite(masks / f"{roi}.tiff", mask)
        for marker, image in channels.items():
            tifffile.imwrite(images / roi / f"{marker}.tiff", image)
obs = pd.DataFrame(rows, index=[f"cell_{i:03d}" for i in range(len(rows))])
obs["parent"] = pd.Categorical(obs.parent)
adata = ad.AnnData(np.zeros((len(obs),2), dtype="float32"), obs=obs,
                   var=pd.DataFrame(index=["IBA1", "SOX2"]))
original_obs = adata.obs.copy(deep=True)


## Classes and image features

Freeze the whole target population per ROI. Full masks remain the source of
boundaries and local-background exclusion. The same worker runs in the GUI or
through `sbt run cellfeat`.

In [2]:
spec = pr.SplitSpec(population_key="parent", populations=("mixed",), classes=(
    pr.ClassSpec(class_id="myeloid", name="Myeloid", source_labels=("Myeloid",)),
    pr.ClassSpec(class_id="glial", name="Glial", source_labels=("Glial",)),
))
experiment = pr.create_feature_experiment(
    adata, spec, directory=workspace / "experiment", images=[images], masks=masks,
    recipe=pr.image_recipe(["IBA1", "SOX2"]),
)
features = pr.build_feature_table(experiment, workers=1, progress=lambda event: None)
display(features.head())


,obs_name,source_population,ROI,ObjectNumber,source::imc::X_loc,source::imc::Y_loc,source::imc::measurement_mask_offset_px,source::imc::measurement_allows_cell_overlap,source::imc::measurement_region_vanished,source::imc::mask_area,...,source::imc::channel::SOX2::q75,source::imc::channel::SOX2::q90,source::imc::channel::SOX2::iqr,source::imc::channel::SOX2::core_mean,source::imc::channel::SOX2::border_mean,source::imc::channel::SOX2::core_to_border_ratio,source::imc::channel::SOX2::weighted_centroid_offset_fraction_radius,source::imc::channel::SOX2::foreground_to_bg_ratio,source::imc::channel::SOX2::foreground_bg_contrast,source::imc::channel::SOX2::foreground_bg_contrast_z
0,cell_000,mixed,case0_roi0,1,6.5,6.5,0,False,False,64.0,...,0.827676,0.839334,0.026674,0.816864,0.812866,1.004918,0.057254,54.203815,0.800077,93.943681
1,cell_001,mixed,case0_roi0,2,21.5,6.5,0,False,False,64.0,...,0.125501,0.132418,0.021216,0.112205,0.114580,0.979275,0.034967,7.525580,0.098196,11.933772
2,cell_002,mixed,case0_roi0,3,36.5,6.5,0,False,False,64.0,...,0.828355,0.841453,0.030350,0.814335,0.810895,1.004243,0.062348,54.245083,0.797845,96.189912
3,cell_003,mixed,case0_roi0,4,51.5,6.5,0,False,False,64.0,...,0.126343,0.134538,0.022898,0.116626,0.109591,1.064194,0.044181,7.676956,0.098758,11.576723
4,cell_004,mixed,case0_roi0,5,6.5,21.5,0,False,False,64.0,...,0.826275,0.836888,0.022789,0.815459,0.813359,1.002582,0.010374,55.768056,0.799934,94.014328


## Marker agreement and sampling

Require IBA1 signal for one class, use SOX2 as supportive evidence for the other.
Feature-only markers need no agreement check. These bounds are meaningful only
for this deliberately constructed synthetic example.

In [3]:
source = pr.MaxFuseSource(label_key="atlas_label", score_key="maxfuse_score")
candidates = source.candidates(adata, spec)
evidence = features.set_index("obs_name")
rules = (
    pr.MarkerRule(class_id="myeloid", feature="source::imc::channel::IBA1::mean", role="required", minimum=.5),
    pr.MarkerRule(class_id="glial", feature="source::imc::channel::SOX2::mean", role="supportive", minimum=.5),
    pr.MarkerRule(class_id="myeloid", feature="source::imc::mask_area", role="feature_only"),
)
selection = pr.select_exemplars(candidates, class_ids=spec.class_ids, evidence=evidence, rules=rules,
    sampling=pr.SamplingSpec(per_class=30, minimum_per_class=10, max_per_stratum=5, seed=42))
display(selection.coverage)
print(selection.warnings)


,class_id,Case,ROI,available,eligible,selected,median_source_score
0,glial,case0,case0_roi0,8,8,5,0.438827
1,glial,case0,case0_roi1,8,8,5,0.567732
2,glial,case1,case1_roi0,8,8,5,0.220109
3,glial,case1,case1_roi1,8,8,5,0.512992
4,glial,case2,case2_roi0,8,8,5,0.446839
5,glial,case2,case2_roi1,8,8,5,0.498492
6,myeloid,case0,case0_roi0,8,8,5,0.481819
7,myeloid,case0,case0_roi1,8,8,5,0.513346
8,myeloid,case1,case1_roi0,8,8,5,0.349239
9,myeloid,case1,case1_roi1,8,8,5,0.507799


[]


## Grouped evaluation

Hold out cases, not individual cells. These metrics describe agreement with
exemplar labels. Independent image review is still required on real data.

In [4]:
columns = pr.image_feature_columns(features)
validation = pr.evaluate_refinement(features, selection, spec, group_key="Case",
    feature_sets={"means": pr.image_feature_columns(features, means_only=True), "image": columns},
    model_types=("random_forest", "hist_gradient_boosting"), seed=42)
display(validation.metrics)


,held_out_group,group_key,feature_set,model,train_cells,test_cells,scorable_fraction,feature_count,balanced_accuracy,macro_f1,accepted_fraction,accepted_accuracy
0,case0,Case,means,random_forest,40,20,1.0,2,1.0,1.0,1.0,1.0
1,case0,Case,means,hist_gradient_boosting,40,20,1.0,2,1.0,1.0,1.0,1.0
2,case0,Case,image,random_forest,40,20,1.0,32,1.0,1.0,1.0,1.0
3,case0,Case,image,hist_gradient_boosting,40,20,1.0,32,1.0,1.0,1.0,1.0
4,case1,Case,means,random_forest,40,20,1.0,2,1.0,1.0,1.0,1.0
5,case1,Case,means,hist_gradient_boosting,40,20,1.0,2,1.0,1.0,1.0,1.0
6,case1,Case,image,random_forest,40,20,1.0,32,1.0,1.0,1.0,1.0
7,case1,Case,image,hist_gradient_boosting,40,20,1.0,32,1.0,1.0,1.0,1.0
8,case2,Case,means,random_forest,40,20,1.0,2,1.0,1.0,1.0,1.0
9,case2,Case,means,hist_gradient_boosting,40,20,1.0,2,1.0,1.0,1.0,1.0


## Fit, save and integrate proposals

No exemplar is treated as human-confirmed. Rejected cells retain the parent
label. Integration returns a table; applying a new observation is a separate
in-memory operation.

In [5]:
result = pr.fit_refinement(features, selection, spec, feature_columns=columns, seed=42)
writer = result.save(workspace / "review")
validation.save(writer)
integrated = result.integrate(adata, output_key="refined_population")
pd.testing.assert_frame_equal(adata.obs, original_obs)
display(result.assignments.groupby(["assignment_source", "class_id"], dropna=False).size())
result.integrate(adata, output_key="refined_population", apply=True)
assert "refined_population" in adata.obs
assert not result.assignments.assignment_source.eq("confirmed").any()
print("Source population preserved; proposed child labels added in memory.")


assignment_source  class_id
model              glial       48
                   myeloid     48
dtype: int64

Source population preserved; proposed child labels added in memory.
